In [1]:
"""
Build assumption_master sheet: product_limits
-----
Extracts per-product × per-country scalar limits (Max age, Max Cover,
Max Multiple, EOH max, retention limits, etc.) from Basis and
Assumptions.xlsx > Assumptions sheet.

Products in scope: gla, lsdb, ci, ci_existing, ci_new, spouse_gla,
spouse_lsdb, ti, adb, accdb, dismem. EAC params live in globals.

Author: Mohamed Hussein Gadiwala
Basis version: A2026_07 (assumption master)
"""

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib
import csv
import openpyxl
import polars as pl

# --- Paths (mirror basis_master structure) ---
BASE = Path(r"J:\Product & Client Solutions\Toolbox\Calculators"
            r"\Business Tools\Tender Summary\GAP Pricing Engine")

SOURCE_XLSX = BASE / "Basis and Assumptions.xlsx"
SOURCE_SHEET = "Assumptions"

ASSUMPTION_VERSION = "A2026_07"
SNAPSHOT_DIR = BASE / "assumption_snapshots" / f"assumption_{ASSUMPTION_VERSION}"
MASTER_XLSX = BASE / "assumption_master" / f"assumption_master_{ASSUMPTION_VERSION}.xlsx"
BUILD_LOG = SNAPSHOT_DIR / "build_log.csv"

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_XLSX.parent.mkdir(parents=True, exist_ok=True)

SHEET_NAME = "product_limits"
PARQUET_PATH = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"

BUILT_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
BUILT_BY = "Mohamed Hussein Gadiwala"

# Load source (data_only=True → get computed values, not formulas)
wb = openpyxl.load_workbook(SOURCE_XLSX, data_only=True)
ws = wb[SOURCE_SHEET]
SOURCE_SHA = hashlib.sha256(SOURCE_XLSX.read_bytes()).hexdigest()

print(f"Building: {SHEET_NAME}")
print(f"Source:   {SOURCE_XLSX.name} > {SOURCE_SHEET}")
print(f"Target:   {PARQUET_PATH}")
print(f"Loaded:   {ws.max_row} rows × {ws.max_column} cols")
print(f"Built at: {BUILT_AT}")

Building: product_limits
Source:   Basis and Assumptions.xlsx > Assumptions
Target:   J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\assumption_snapshots\assumption_A2026_07\product_limits.parquet
Loaded:   1079 rows × 30 cols
Built at: 2026-07-22T10:38:08+00:00


In [2]:
def find_all_cells_any_col(ws, label: str, max_row: int = 1200, max_col: int = 30) -> list[tuple[int, int]]:
    """Find ALL cells matching `label` exactly, anywhere on the sheet.
    Returns (row, col) tuples, 1-indexed."""
    hits = []
    for row in range(1, max_row + 1):
        for col in range(1, max_col + 1):
            v = ws.cell(row=row, column=col).value
            if isinstance(v, str) and v.strip() == label:
                hits.append((row, col))
    return hits

def find_in_range(ws, label: str, row_start: int, row_end: int, col: int | None = None) -> tuple[int, int] | None:
    """Find first cell matching `label` within given row range (and optional col)."""
    hits = find_all_cells_any_col(ws, label)
    for r, c in hits:
        if row_start <= r < row_end and (col is None or c == col):
            return (r, c)
    return None

def get_col_letter(col: int) -> str:
    return openpyxl.utils.get_column_letter(col)

In [3]:
# =============================================================================
# Product-specific harvesters
# =============================================================================

COUNTRIES = ["SA", "NAM", "eSw"]

def harvest_gla(ws) -> list[dict]:
    """GLA: 3-country grid (SA/NAM/eSw) for 4 params, plus 3 flat params."""
    rows = []

    # Grid params: label in col B, values in cols C/D/E for SA/NAM/eSw
    GLA_GRID = [
        ("Max CTC Age",                        "max_ctc_age",
         "GLA-specific CTC age ceiling (source label preserved)"),
        ("EOH max (GLA)",                      "eoh_max",
         "Evidence of Health max for GLA"),
        ("Max Multiple",                       "max_multiple",
         "Max multiple of salary"),
        ("OM Reinsurance GLA Retention Limit", "om_reinsurance_retention_limit",
         "Old Mutual reinsurance retention limit"),
    ]
    for source_label, canonical, notes in GLA_GRID:
        hit = find_in_range(ws, source_label, 202, 250)
        if hit is None:
            raise RuntimeError(f"GLA grid anchor not found: {source_label!r}")
        r, c = hit
        for i, country in enumerate(COUNTRIES):
            val = ws.cell(row=r, column=c + 1 + i).value
            rows.append({
                "product": "gla", "country": country,
                "param_name": canonical, "param_value": val,
                "source_label": source_label, "source_row": r, "notes": notes,
            })

    # Flat params (country="all")
    GLA_FLAT = [
        ("Max CO Age",                       "max_co_age",
         "Max age for Conversion Option"),
        ("Withdrawal only pre NRA Year CAP", "withdrawal_only_pre_nra_cap",
         "Year cap for withdrawal-only elections pre-NRA"),
        ("GLA Treaty aggregate Limit",       "treaty_aggregate_limit",
         "GLA reinsurance treaty aggregate limit"),
    ]
    for source_label, canonical, notes in GLA_FLAT:
        hit = find_in_range(ws, source_label, 202, 250)
        if hit is None:
            raise RuntimeError(f"GLA flat anchor not found: {source_label!r}")
        r, c = hit
        val = ws.cell(row=r, column=c + 1).value
        rows.append({
            "product": "gla", "country": "all",
            "param_name": canonical, "param_value": val,
            "source_label": source_label, "source_row": r, "notes": notes,
        })

    return rows


def harvest_simple_section(
    ws, product: str, row_start: int, row_end: int,
    param_map: list[tuple[str, str, str]],
) -> list[dict]:
    """For products with flat label→value pairs (label in col B, value in col C).
    param_map: list of (source_label, canonical_name, notes)."""
    rows = []
    for source_label, canonical, notes in param_map:
        hit = find_in_range(ws, source_label, row_start, row_end)
        if hit is None:
            raise RuntimeError(
                f"{product} anchor not found in rows {row_start}-{row_end}: {source_label!r}"
            )
        r, c = hit
        val = ws.cell(row=r, column=c + 1).value
        rows.append({
            "product": product, "country": "all",
            "param_name": canonical, "param_value": val,
            "source_label": source_label, "source_row": r, "notes": notes,
        })
    return rows


def harvest_ci(ws) -> list[dict]:
    """CI: 2-column grid (Existing/New) at cols F/G rows 317-320, plus 3 flat CI-wide."""
    rows = []

    # Grid label "Critical Incidents Limits" at col E. Params labelled in col E,
    # Existing values in col F, New values in col G.
    CI_GRID = [
        ("Max age",       "max_age",     "CI max age"),
        ("Max Cover",     "max_cover",   "CI max cover"),
        ("Max Multiple",  "max_multiple","CI max multiple"),
        ("EOH max (CI)",  "eoh_max_ci",  "CI Evidence of Health max"),
    ]
    for source_label, canonical, notes in CI_GRID:
        hit = find_in_range(ws, source_label, 313, 343, col=5)  # col E = 5
        if hit is None:
            raise RuntimeError(f"CI grid anchor not found in col E: {source_label!r}")
        r, c = hit
        for variant_offset, variant_key in [(1, "ci_existing"), (2, "ci_new")]:
            val = ws.cell(row=r, column=c + variant_offset).value
            rows.append({
                "product": variant_key, "country": "all",
                "param_name": canonical, "param_value": val,
                "source_label": source_label, "source_row": r, "notes": notes,
            })

    # CI-wide flat params (apply to both variants → use bare "ci" key)
    CI_FLAT = [
        ("New Products Retention Limits", "new_products_retention_limit",
         "CI reinsurance retention limit (new products)"),
        ("Max CO Age",                     "max_co_age",
         "CI conversion option max age"),
        ("Withdrawal only pre NRA Year CAP","withdrawal_only_pre_nra_cap",
         "CI withdrawal-only year cap"),
    ]
    for source_label, canonical, notes in CI_FLAT:
        hit = find_in_range(ws, source_label, 313, 343)
        if hit is None:
            raise RuntimeError(f"CI flat anchor not found: {source_label!r}")
        r, c = hit
        val = ws.cell(row=r, column=c + 1).value
        rows.append({
            "product": "ci", "country": "all",
            "param_name": canonical, "param_value": val,
            "source_label": source_label, "source_row": r, "notes": notes,
        })

    return rows


def harvest_spouses(ws) -> list[dict]:
    """Spouses Cover section holds TWO products: Spouse's GLA (346-349) and
    Spouse's LSDB (351-354). Same param labels, different row ranges."""
    rows = []
    SPOUSE_PARAMS = [
        ("Max age",      "max_age",      ""),
        ("Max Cover",    "max_cover",    ""),
        ("Max Multiple", "max_multiple", ""),
    ]
    for product_key, (r_start, r_end), notes_prefix in [
        ("spouse_gla",  (346, 350), "Spouse's GLA"),
        ("spouse_lsdb", (351, 355), "Spouse's LSDB"),
    ]:
        for source_label, canonical, _ in SPOUSE_PARAMS:
            hit = find_in_range(ws, source_label, r_start, r_end)
            if hit is None:
                raise RuntimeError(
                    f"{product_key} anchor not found rows {r_start}-{r_end}: {source_label!r}"
                )
            r, c = hit
            val = ws.cell(row=r, column=c + 1).value
            rows.append({
                "product": product_key, "country": "all",
                "param_name": canonical, "param_value": val,
                "source_label": source_label, "source_row": r,
                "notes": notes_prefix,
            })
    return rows

## Edit 2 — Cell 2: add `harvest_fc()` after `harvest_spouses()`

def harvest_fc(ws) -> list:
    """FC: 5 cost/loading scalars from Assumptions!C427:C431. All country='all'.

    Note: FC children rate SCHEDULE (D4/D5/E5 on 'FC Basis - New') is a rate
    table and lives in the fc_rates builder (basis snapshot), not here.
    """
    rows = []

    FC_FLAT = [
        # (source_label,             canonical,                notes)
        ("CTC Cost",                 "ctc_cost",               "FC Cover-to-Continue cost loading"),
        ("Death and LSDB PUP Cost",  "pup_cost_death_lsdb",    "FC PUP cost when both Death and LSDB elected"),
        ("Death PUP Cost",           "pup_cost_death",         "FC PUP cost when Death only elected"),
        ("LSDB PUP Cost",            "pup_cost_lsdb",          "FC PUP cost when LSDB only elected"),
        ("% second spouse",          "second_spouse_loading",  "Additional loading applied per second spouse"),
    ]

    for source_label, canonical, notes in FC_FLAT:
        hit = find_in_range(ws, source_label, 425, 435)
        if hit is None:
            raise RuntimeError(f"FC flat anchor not found: {source_label!r}")
        r, c = hit
        val = ws.cell(row=r, column=c + 1).value
        rows.append({
            "product": "fc",
            "country": "all",
            "param_name": canonical,
            "param_value": val,
            "source_label": source_label,
            "source_row": r,
            "notes": notes,
        })

    return rows

# =============================================================================
# Execute all harvesters
# =============================================================================

all_rows: list[dict] = []

# GLA
all_rows.extend(harvest_gla(ws))

# LSDB
all_rows.extend(harvest_simple_section(ws, "lsdb", 250, 313, [
    ("Max age",                            "max_age",
     "LSDB max age"),
    ("Max Cover",                          "max_cover",
     "LSDB max cover"),
    ("Max Multiple",                       "max_multiple",
     "LSDB max multiple"),
    ("Max CO Age",                         "max_co_age",
     "LSDB conversion option max age"),
    ("OM Reinsurance LSDB Retention Limit","om_reinsurance_retention_limit",
     "LSDB reinsurance retention"),
    ("Allow 0 reduction period",           "allow_zero_reduction_period",
     "LSDB allows zero reduction period (Y/N)"),
    ("Allow 0 month waiting period",       "allow_zero_waiting_period",
     "LSDB allows zero waiting period (Y/N)"),
]))

# CI (variant grid + shared flats)
all_rows.extend(harvest_ci(ws))

# Spouses (compound section → 2 products)
all_rows.extend(harvest_spouses(ws))

# TI
all_rows.extend(harvest_simple_section(ws, "ti", 371, 380, [
    ("Max age",         "max_age",      "TI max age"),
    ("Max Cover",       "max_cover",    "TI max cover"),
    ("Max Multiple",    "max_multiple", "TI max multiple"),
    ("Allow LSDB + TI?","allow_lsdb_ti",
     "Can TI stack with LSDB. Source stores as text ('false'/'true'); "
     "pricing engine must coerce."),
]))

# ADB (Accidental Death)
all_rows.extend(harvest_simple_section(ws, "adb", 380, 389, [
    ("Max age",                "max_age",                 "Acc Death max age"),
    ("Max Cover",              "max_cover",               "Acc Death max cover"),
    ("Max Multiple",           "max_multiple",            "Acc Death max multiple"),
    ("CtC on AccDth indicator","ctc_on_accdth_indicator", "Cost-to-Company on Acc Death flag"),
]))

# ACCDB (Accidental Disability)
all_rows.extend(harvest_simple_section(ws, "accdb", 389, 398, [
    ("Max age",      "max_age",      "Acc Disability max age"),
    ("Max Cover",    "max_cover",    "Acc Disability max cover"),
    ("Max Multiple", "max_multiple", "Acc Disability max multiple"),
]))

# DISMEM
all_rows.extend(harvest_simple_section(ws, "dismem", 398, 407, [
    ("Max age",      "max_age",      "Dismemberment max age"),
    ("Max Cover",    "max_cover",    "Dismemberment max cover"),
    ("Max Multiple", "max_multiple", "Dismemberment max multiple"),
]))

# FC (Family Cover) scalars — children rate schedule is in fc_rates builder
all_rows.extend(harvest_fc(ws))

# --- Summary ---
product_counts = Counter(r["product"] for r in all_rows)
print(f"Total rows harvested: {len(all_rows)}")
print(f"\nRows per product:")
for prod, cnt in sorted(product_counts.items()):
    print(f"  {prod:15s}: {cnt}")

print(f"\nFirst 5 rows preview (raw types):")
for r in all_rows[:5]:
    print(f"  {r['product']:12s} | {r['country']:4s} | {r['param_name']:35s} = "
          f"{r['param_value']!r} ({type(r['param_value']).__name__})")

Total rows harvested: 58

Rows per product:
  accdb          : 3
  adb            : 4
  ci             : 3
  ci_existing    : 4
  ci_new         : 4
  dismem         : 3
  fc             : 5
  gla            : 15
  lsdb           : 7
  spouse_gla     : 3
  spouse_lsdb    : 3
  ti             : 4

First 5 rows preview (raw types):
  gla          | SA   | max_ctc_age                         = 65 (int)
  gla          | NAM  | max_ctc_age                         = 65 (int)
  gla          | eSw  | max_ctc_age                         = 65 (int)
  gla          | SA   | eoh_max                             = 16800000 (int)
  gla          | NAM  | eoh_max                             = 16800000 (int)


In [4]:
# =============================================================================
# CRITICAL: classify types BEFORE polars sees the mixed-type column.
# Polars upcasts mixed-type columns to Utf8 at construction time, which would
# make classify() return 'string' for every value. So we classify in Python
# first, then hand polars a clean pre-stringified column.
# =============================================================================

def classify(v) -> str:
    if v is None:            return "null"
    if isinstance(v, bool):  return "bool"       # must come before int (bool ⊂ int in Python)
    if isinstance(v, int):   return "int"
    if isinstance(v, float): return "float"
    return "string"

# Enrich the raw rows with param_type + stringified value
for r in all_rows:
    v = r["param_value"]
    r["param_type"] = classify(v)
    r["param_value_str"] = "" if v is None else str(v)

# Build DataFrame with canonical column order
df = pl.DataFrame(all_rows).select([
    "product",
    "country",
    "param_name",
    pl.col("param_value_str").alias("param_value"),
    "param_type",
    "source_label",
    "source_row",
    "notes",
])

# =============================================================================
# Structural validation
# =============================================================================
key_cols = ["product", "country", "param_name"]
dupes = df.group_by(key_cols).len().filter(pl.col("len") > 1)
assert dupes.height == 0, f"Duplicate keys found:\n{dupes}"

expected_products = {
    "gla", "lsdb", "ci", "ci_existing", "ci_new",
    "spouse_gla", "spouse_lsdb", "ti", "adb", "accdb", "dismem",
    "fc",   # added 2026-07-22
}
actual_products = set(df["product"].unique().to_list())
unexpected = actual_products - expected_products
missing = expected_products - actual_products
assert not unexpected, f"Unexpected products: {unexpected}"
assert not missing,    f"Missing products: {missing}"

# =============================================================================
# Frozen value expectations for A2026_07
# =============================================================================
def get_val(product: str, country: str, param: str) -> tuple[str, str]:
    row = df.filter(
        (pl.col("product") == product)
        & (pl.col("country") == country)
        & (pl.col("param_name") == param)
    )
    assert row.height == 1, f"Missing: {product}/{country}/{param}"
    return row["param_value"][0], row["param_type"][0]

def assert_numeric(product, country, param, expected_val):
    actual_str, actual_type = get_val(product, country, param)
    assert actual_type in ("int", "float"), (
        f"{product}/{country}/{param}: expected numeric, got type {actual_type!r}"
    )
    actual = float(actual_str)
    assert abs(actual - float(expected_val)) < 1e-6, (
        f"{product}/{country}/{param}: got {actual}, expected {expected_val}"
    )

# Numeric frozen checks
frozen_numeric = [
    ("gla",         "SA",  "max_ctc_age",                     65),
    ("gla",         "NAM", "eoh_max",                         16_800_000),
    ("gla",         "eSw", "om_reinsurance_retention_limit",  8_000_000),
    ("gla",         "all", "max_co_age",                      80),
    ("gla",         "all", "treaty_aggregate_limit",          10_000_000_000),
    ("lsdb",        "all", "max_cover",                       8_500_000),
    ("ci_existing", "all", "max_cover",                       1_750_000),
    ("ci_new",      "all", "max_cover",                       3_000_000),
    ("ci",          "all", "new_products_retention_limit",    1_500_000),
    ("spouse_gla",  "all", "max_age",                         99),
    ("spouse_lsdb", "all", "max_cover",                       1_000_000),
    ("ti",          "all", "max_cover",                       45_000_000),
    ("adb",         "all", "max_age",                         89),
    ("dismem",      "all", "max_multiple",                    5),
        # FC — 5 scalars from Assumptions!C427:C431
    ("fc", "all", "ctc_cost",              0.0),
    ("fc", "all", "pup_cost_death_lsdb",   0.17),
    ("fc", "all", "pup_cost_death",        0.1),
    ("fc", "all", "pup_cost_lsdb",         0.1),
    ("fc", "all", "second_spouse_loading", 0.1),

]
for args in frozen_numeric:
    assert_numeric(*args)

# LSDB Y/N flags — stored as strings in source, preserved as-is (rule 5.4)
for param in ("allow_zero_waiting_period", "allow_zero_reduction_period"):
    val, typ = get_val("lsdb", "all", param)
    assert typ == "string", f"lsdb/{param}: expected string, got {typ}"
    assert val == "Y",      f"lsdb/{param}: expected 'Y', got {val!r}"

# TI allow_lsdb_ti — source stores as text 'false' (not a real bool). Rule 5.4.
val, typ = get_val("ti", "all", "allow_lsdb_ti")
assert typ in ("bool", "string", "int"), f"allow_lsdb_ti unexpected type: {typ}"
assert val.lower() in ("false", "true", "0", "1"), (
    f"allow_lsdb_ti unexpected value: {val!r}"
)

print(f"✓ {df.height} rows validated")
print(f"✓ {len(expected_products)} products present")
print(f"✓ {len(frozen_numeric)} numeric frozen checks passed")
print(f"✓ LSDB Y/N flags preserved as strings")
print(f"✓ TI allow_lsdb_ti preserved as source value {val!r} (type={typ})")

# --- Type distribution report ---
type_dist = df.group_by("param_type").len().sort("param_type")
print(f"\nparam_type distribution:")
print(type_dist)

print(f"\nFinal DataFrame:")
print(df)

✓ 58 rows validated
✓ 12 products present
✓ 19 numeric frozen checks passed
✓ LSDB Y/N flags preserved as strings
✓ TI allow_lsdb_ti preserved as source value 'False' (type=bool)

param_type distribution:
shape: (4, 2)
┌────────────┬─────┐
│ param_type ┆ len │
│ ---        ┆ --- │
│ str        ┆ u32 │
╞════════════╪═════╡
│ bool       ┆ 1   │
│ float      ┆ 4   │
│ int        ┆ 51  │
│ string     ┆ 2   │
└────────────┴─────┘

Final DataFrame:
shape: (58, 8)
┌─────────┬─────────┬─────────────┬────────────┬────────────┬────────────┬────────────┬────────────┐
│ product ┆ country ┆ param_name  ┆ param_valu ┆ param_type ┆ source_lab ┆ source_row ┆ notes      │
│ ---     ┆ ---     ┆ ---         ┆ e          ┆ ---        ┆ el         ┆ ---        ┆ ---        │
│ str     ┆ str     ┆ str         ┆ ---        ┆ str        ┆ ---        ┆ i64        ┆ str        │
│         ┆         ┆             ┆ str        ┆            ┆ str        ┆            ┆            │
╞═════════╪═════════╪════════════

In [5]:
# --- Write immutable parquet snapshot ---
df.write_parquet(PARQUET_PATH)
parquet_bytes = PARQUET_PATH.read_bytes()
PARQUET_SHA = hashlib.sha256(parquet_bytes).hexdigest()
PARQUET_SIZE_KB = len(parquet_bytes) / 1024

print(f"✓ Wrote {PARQUET_PATH.name}")
print(f"  Size:    {PARQUET_SIZE_KB:.2f} KB")
print(f"  SHA-256: {PARQUET_SHA[:16]}...")

# --- Append/replace sheet in master workbook ---
if MASTER_XLSX.exists():
    master_wb = openpyxl.load_workbook(MASTER_XLSX)
else:
    master_wb = openpyxl.Workbook()
    if "Sheet" in master_wb.sheetnames:
        del master_wb["Sheet"]

if SHEET_NAME in master_wb.sheetnames:
    del master_wb[SHEET_NAME]

ws_out = master_wb.create_sheet(SHEET_NAME)
ws_out.append(df.columns)
for row in df.iter_rows():
    ws_out.append(list(row))

master_wb.save(MASTER_XLSX)
print(f"✓ Master workbook updated: {MASTER_XLSX.name}")

✓ Wrote product_limits.parquet
  Size:    4.90 KB
  SHA-256: 29e2596743375e1c...
✓ Master workbook updated: assumption_master_A2026_07.xlsx


In [6]:
LOG_HEADERS = [
    "built_at", "built_by", "sheet", "rows", "cols",
    "parquet_path", "parquet_size_kb", "parquet_sha256",
    "source_file", "source_sheet", "source_sha256",
    "assumption_version", "notes",
]

log_row = {
    "built_at": BUILT_AT,
    "built_by": BUILT_BY,
    "sheet": SHEET_NAME,
    "rows": df.height,
    "cols": len(df.columns),
    "parquet_path": str(PARQUET_PATH.relative_to(BASE)),
    "parquet_size_kb": f"{PARQUET_SIZE_KB:.2f}",
    "parquet_sha256": PARQUET_SHA,
    "source_file": SOURCE_XLSX.name,
    "source_sheet": SOURCE_SHEET,
    "source_sha256": SOURCE_SHA,
    "assumption_version": ASSUMPTION_VERSION,
    "notes": (
    "Product limits. 12 product keys "
    "(ci split into ci/ci_existing/ci_new; fc added 2026-07-22). "
    "GLA has 3-country grid (SA/NAM/eSw). EAC skipped (params in globals). "
    "FC has 5 cost/loading scalars; children rate schedule lives in fc_rates. "
    "Type classification done in Python before polars upcast. "
    "TI allow_lsdb_ti stored as text 'false' in source, preserved verbatim."
),
}

write_header = not BUILD_LOG.exists()
with BUILD_LOG.open("a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=LOG_HEADERS)
    if write_header:
        writer.writeheader()
    writer.writerow(log_row)

print(f"✓ build_log.csv updated ({BUILD_LOG.name})")
print(f"\n{'='*60}")
print(f"BUILD COMPLETE: {SHEET_NAME}")
print(f"{'='*60}")

✓ build_log.csv updated (build_log.csv)

BUILD COMPLETE: product_limits
